In [ ]:
# Cell 1: load overlay and configure shared parameters.
from time import sleep, time
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

BITFILE = 'base_add.bit'
PL_CLK_HZ = 125_000_000
SENTINEL = np.uint32(0xDEADBEEF)

# Board IO register offsets in led_ctrl_0.
LED_CTRL = 0x00
LED_SPEED_DIV = 0x04
LED_VALUE = 0x08
LED_STATUS = 0x0C

# ADC capture register offsets in adc_capture_0.
CTRL = 0x00
STATUS = 0x04
SAMPLE_COUNT = 0x08
ADC_HALF = 0x0C
SAMPLE_DELAY = 0x10
DECIMATION = 0x14
CHANNEL_MASK = 0x18
CAPTURE_MODE = 0x1C
TRIGGER_MODE = 0x20
PRE_DELAY = 0x24
BUFFER_SELECT = 0x28
LATEST_A = 0x2C
LATEST_B = 0x30
SAMPLE_COUNTER = 0x34
FIFO_LEVEL = 0x38
ERROR_FLAGS = 0x3C
VERSION = 0x44
SAVED_COUNTER = 0x48
LAST_AXIS_WORD = 0x4C
DEBUG_STATE = 0x50
AXIS_SENT_COUNT = 0x54
AXIS_STALL_COUNT = 0x58
TLAST_COUNT = 0x5C
FIFO_BACKPRESSURE = 0x60
DROPPED_SAMPLE_COUNT = 0x64
CAPTURE_DONE_LATCHED = 0x68
CORE_DONE = 0x6C

S2MM_DMASR = 0x34

COLOR_NAME = {
    0: 'OFF',
    1: 'BLUE',
    2: 'GREEN',
    3: 'CYAN',
    4: 'RED',
    5: 'MAGENTA',
    6: 'YELLOW',
    7: 'WHITE',
}

def find_ip(overlay, text):
    matches = [name for name in overlay.ip_dict.keys() if text in name.lower()]
    if not matches:
        raise RuntimeError('Cannot find IP containing %r. IPs: %s' % (text, list(overlay.ip_dict.keys())))
    name = matches[0]
    return name, getattr(overlay, name)

def set_board_io(led_mask=0, ld4_color=0, ld5_color=0):
    led_mask &= 0xF
    ld4_color &= 0x7
    ld5_color &= 0x7
    value = led_mask | (ld4_color << 4) | (ld5_color << 7)
    led_ip.write(LED_CTRL, 0x00)
    led_ip.write(LED_VALUE, value)
    return value

def read_board_io():
    status = led_ip.read(LED_STATUS)
    return {
        'raw': status,
        'leds': status & 0xF,
        'ld4_color': (status >> 4) & 0x7,
        'ld5_color': (status >> 7) & 0x7,
        'buttons': (status >> 10) & 0xF,
    }

def dump_capture_regs():
    print('STATUS          = 0x%08X' % adc_ip.read(STATUS))
    print('ERROR_FLAGS     = 0x%08X' % adc_ip.read(ERROR_FLAGS))
    print('LATEST_A/B      = %d / %d' % (adc_ip.read(LATEST_A), adc_ip.read(LATEST_B)))
    print('SAMPLE_COUNTER  = %d' % adc_ip.read(SAMPLE_COUNTER))
    print('FIFO_LEVEL      = %d' % adc_ip.read(FIFO_LEVEL))
    print('SAVED_COUNTER   = %d' % adc_ip.read(SAVED_COUNTER))
    print('LAST_AXIS_WORD  = 0x%08X' % adc_ip.read(LAST_AXIS_WORD))
    print('DEBUG_STATE     = %d' % adc_ip.read(DEBUG_STATE))
    print('AXIS_SENT_COUNT = %d' % adc_ip.read(AXIS_SENT_COUNT))
    print('TLAST_COUNT     = %d' % adc_ip.read(TLAST_COUNT))
    print('DROPPED_SAMPLES = %d' % adc_ip.read(DROPPED_SAMPLE_COUNT))
    print('VERSION         = 0x%08X' % adc_ip.read(VERSION))
    print('S2MM_DMASR      = 0x%08X' % dma.mmio.read(S2MM_DMASR))

def configure_capture(sample_count, adc_half_period, decimation, capture_mode, sample_delay=1):
    adc_ip.write(CTRL, 0x04)
    adc_ip.write(CTRL, 0x00)
    adc_ip.write(ERROR_FLAGS, 0xFFFFFFFF)
    adc_ip.write(SAMPLE_COUNT, int(sample_count))
    adc_ip.write(ADC_HALF, int(adc_half_period))
    adc_ip.write(SAMPLE_DELAY, int(sample_delay))
    adc_ip.write(DECIMATION, int(decimation))
    adc_ip.write(CHANNEL_MASK, 0b11)
    adc_ip.write(CAPTURE_MODE, int(capture_mode))
    adc_ip.write(TRIGGER_MODE, 0)
    adc_ip.write(PRE_DELAY, 0)
    adc_ip.write(BUFFER_SELECT, 0)

def capture_once(sample_count, adc_half_period=1, decimation=1, capture_mode=2, sample_delay=1):
    sample_count = int(sample_count)
    buf = allocate(shape=(sample_count,), dtype=np.uint32)
    buf[:] = SENTINEL
    buf.flush()
    dma.recvchannel.transfer(buf)
    configure_capture(sample_count, adc_half_period, decimation, capture_mode, sample_delay)
    adc_ip.write(CTRL, 0x01)
    adc_ip.write(CTRL, 0x03)
    adc_ip.write(CTRL, 0x01)
    t0 = time()
    dma.recvchannel.wait()
    elapsed = time() - t0
    buf.invalidate()
    raw = np.array(buf, dtype=np.uint32)
    ch0 = raw & np.uint32(0x0FFF)
    ch1 = (raw >> np.uint32(16)) & np.uint32(0x0FFF)
    if np.any(raw == SENTINEL):
        raise RuntimeError('DMA buffer still contains sentinel values.')
    if adc_ip.read(AXIS_SENT_COUNT) != sample_count:
        raise RuntimeError('AXIS_SENT_COUNT mismatch: %d != %d' % (adc_ip.read(AXIS_SENT_COUNT), sample_count))
    if adc_ip.read(TLAST_COUNT) != 1:
        raise RuntimeError('TLAST_COUNT is not 1.')
    if adc_ip.read(DROPPED_SAMPLE_COUNT) != 0:
        raise RuntimeError('Dropped samples detected.')
    if adc_ip.read(STATUS) & (1 << 10):
        raise RuntimeError('STATUS.fatal_error is set.')
    return raw, ch0, ch1, elapsed

overlay = Overlay(BITFILE)
led_name, led_ip = find_ip(overlay, 'led_ctrl')
adc_name, adc_ip = find_ip(overlay, 'adc_capture')
dma_name, dma = find_ip(overlay, 'dma')

set_board_io(0, 0, 0)
print('Overlay loaded:', BITFILE)
print('LED/RGB/BTN IP:', led_name, hex(overlay.ip_dict[led_name]['phys_addr']))
print('ADC IP        :', adc_name, hex(overlay.ip_dict[adc_name]['phys_addr']))
print('DMA IP        :', dma_name, hex(overlay.ip_dict[dma_name]['phys_addr']))
print('RGB color map :', COLOR_NAME)


In [ ]:
# Cell 2: test 4 buttons, 4 single-color LEDs, and 2 RGB LEDs.
print('Walking LD0..LD3')
for i in range(4):
    value = set_board_io(1 << i, 0, 0)
    sleep(0.5)
    s = read_board_io()
    print('LED bit %d, write=0x%03X, status=%s' % (i, value, s))

print('\nTesting LD4 and LD5 RGB colors')
for code in range(8):
    value = set_board_io(0, code, code)
    sleep(0.45)
    s = read_board_io()
    print('RGB code %d %-7s write=0x%03X status=%s' % (code, COLOR_NAME[code], value, s))

print('\nReading buttons for 8 seconds. Press BTN0..BTN3 and watch the bit mask.')
t_end = time() + 8.0
last = None
while time() < t_end:
    s = read_board_io()
    if s['buttons'] != last:
        print('buttons = 0b%04d raw_status=0x%08X' % (int(bin(s['buttons'])[2:]), s['raw']))
        last = s['buttons']
    sleep(0.05)

set_board_io(0, 0, 0)
print('Board IO test done; outputs off.')


In [ ]:
# Cell 3: test fake stream -> AXIS FIFO -> AXI DMA -> DDR.
FAKE_SAMPLE_COUNT = 65536
raw, ch0, ch1, elapsed = capture_once(
    sample_count=FAKE_SAMPLE_COUNT,
    adc_half_period=1,
    decimation=1,
    capture_mode=2,
    sample_delay=1,
)

expected = np.arange(FAKE_SAMPLE_COUNT, dtype=np.uint32) & np.uint32(0x0FFF)
if not np.array_equal(ch0, expected):
    raise RuntimeError('CH0 fake pattern mismatch.')
if not np.array_equal(ch1, np.uint32(4095) - expected):
    raise RuntimeError('CH1 fake pattern mismatch.')

print('PASS: fake FIFO/DMA path')
print('elapsed = %.6f s' % elapsed)
print('CH0 first 16:', ch0[:16].tolist())
print('CH1 first 16:', ch1[:16].tolist())
dump_capture_regs()

plt.figure(figsize=(10, 4))
plt.plot(ch0[:256], label='fake CH0')
plt.plot(ch1[:256], label='fake CH1')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Cell 4: test real ADC capture.
# For first bring-up, keep this moderate. Increase REAL_SAMPLE_COUNT after waveform looks sane.
REAL_SAMPLE_COUNT = 65536
REAL_ADC_HALF_PERIOD = 3125   # 125 MHz / (2 * 3125) = 20 kS/s, good for 1200/2200 Hz waveform checks
REAL_DECIMATION = 1
REAL_SAMPLE_DELAY = 1

fs_hz = PL_CLK_HZ / (2 * REAL_ADC_HALF_PERIOD * REAL_DECIMATION)
raw, ch0, ch1, elapsed = capture_once(
    sample_count=REAL_SAMPLE_COUNT,
    adc_half_period=REAL_ADC_HALF_PERIOD,
    decimation=REAL_DECIMATION,
    capture_mode=1,
    sample_delay=REAL_SAMPLE_DELAY,
)

print('PASS: real ADC DMA transfer completed')
print('elapsed      = %.6f s' % elapsed)
print('sample rate  = %.3f Hz' % fs_hz)
print('duration     = %.3f s' % (REAL_SAMPLE_COUNT / fs_hz))
print('CH0 mean/Vpp = %.2f / %d' % (float(ch0.mean()), int(ch0.max() - ch0.min())))
print('CH1 mean/Vpp = %.2f / %d' % (float(ch1.mean()), int(ch1.max() - ch1.min())))
dump_capture_regs()

n = min(1000, REAL_SAMPLE_COUNT)
t_ms = np.arange(n) / fs_hz * 1000.0
plt.figure(figsize=(12, 4))
plt.plot(t_ms, ch0[:n] - np.mean(ch0[:n]), label='CH0 DC removed')
plt.plot(t_ms, ch1[:n] - np.mean(ch1[:n]), label='CH1 DC removed', alpha=0.75)
plt.xlabel('Time (ms)')
plt.ylabel('ADC counts, DC removed')
plt.grid(True)
plt.legend()
plt.show()
